In [1]:
import json
import chromadb
from langchain_community.embeddings import OllamaEmbeddings

In [3]:
embeddings = OllamaEmbeddings(model = "nomic-embed-text")
# 2. Initialize persistent ChromaDB - persistent client is saved to disk
chroma_client = chromadb.PersistentClient(path="./gita_chroma_db")

collection = chroma_client.get_or_create_collection(
    name = "bhagavad-gita",
    metadata={"hnsw:space":"cosine"} #using cosine similarity
)

with open('clean_gita_data.json','r',encoding='utf-8') as f:
    gita_data = json.load(f)

documents = []
metadata = []
ids = []

print('preparing data for embedding')

for i in gita_data:
    english_translation = i["english_translation"]
    documents.append(english_translation)

    chapter = i['chapter']
    verse = i['verse']
    speaker = i['speaker']

    metadata.append({"chapter":chapter,"verse":verse,"speaker":speaker})

    unique_id = f"ch{chapter}_v{verse}"
    ids.append(unique_id)

# add to chromadb in batches
print("Generating Embeddings and saving to chromadb")
embedded_docs = embeddings.embed_documents(documents)

collection.add(
    embeddings=embedded_docs,
    documents=documents,
    metadatas=metadata,
    ids=ids
)

print(f"Successfully embedded and stored {collection.count()} verses.")

preparing data for embedding
Generating Embeddings and saving to chromadb
Successfully embedded and stored 701 verses.
